## Criar um algoritmo capaz de prever, usando regressão logística, se os clientes que não avaliaram o produto ficaram ou não satisfeitos com a compra.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("/content/SuperMarket Analysis.csv")
display(df)
df["Satisfaction"] = df["Rating"]>=5
df.info()
display("Linhas Vazias: ", df.isna().sum()[df.isna().sum() > 0])
target = df["Satisfaction"]
colunas_selecionadas = ["Gender", "Unit price", "Quantity", "Sales", "Payment"]
df_treinamento = df[colunas_selecionadas]

df_treinamento_final = pd.get_dummies(df_treinamento, drop_first=True)

display(df_treinamento_final)

FileNotFoundError: [Errno 2] No such file or directory: '/content/SuperMarket Analysis.csv'

In [ ]:
contagem = df["Satisfaction"].value_counts()
plt.figure(figsize=(8, 5))
barras = plt.bar(contagem.index, contagem.values, color=['#4CAF50', '#F44336'])

plt.title('Insatisfeito vs Satisfeito', fontsize=14)
plt.xlabel('Status da Avaliação', fontsize=12)
plt.ylabel('Quantidade de Produtos', fontsize=12)

plt.bar_label(barras, padding=3)

plt.show()

In [ ]:
import pandas as pd

colunas_para_outliers = ["Unit price", "Quantity", "Sales"]
df_numerico = df_treinamento[colunas_para_outliers].copy()

q1 = []
q3 = []
iqr = []
limite_inferior = []
limite_superior = []

for i, coluna in enumerate(df_numerico.columns):
    q1.append(df_numerico[coluna].quantile(0.25))
    q3.append(df_numerico[coluna].quantile(0.75))
    iqr.append(q3[i] - q1[i])

    limite_inferior.append(q1[i] - 1.5 * iqr[i])
    limite_superior.append(q3[i] + 1.5 * iqr[i])

linhas_com_outliers = pd.Series(False, index=df_numerico.index)

for i, coluna in enumerate(df_numerico.columns):
    linhas_com_outliers |= ((df_numerico[coluna] < limite_inferior[i]) | (df_numerico[coluna] > limite_superior[i]))

id_outliers = linhas_com_outliers[linhas_com_outliers].index
outliers_reais = df_numerico[linhas_com_outliers]

print(f"Total de linhas com pelo menos um outlier: {len(outliers_reais)}")
display(df.loc[id_outliers])

for i, coluna in enumerate(df_numerico.columns):
    df_treinamento_final.loc[df[coluna] < limite_inferior[i], coluna] = limite_inferior[i]
    df_treinamento_final.loc[df[coluna] > limite_superior[i], coluna] = limite_superior[i]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_treino, X_teste, y_treino, y_teste = train_test_split(
    df_treinamento_final,
    target,
    test_size = 0.4,
    random_state = 42
)

scaler = StandardScaler()

X_treino_padronizado = scaler.fit_transform(X_treino)
X_teste_padronizado = scaler.transform(X_teste)

modelo_logistico = LogisticRegression()
modelo_logistico.fit(X_treino_padronizado, y_treino)

previsao = modelo_logistico.predict(X_teste_padronizado)
acuracia = accuracy_score(y_teste, previsao)

print(f"Acuracia f{acuracia}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelo_rf = RandomForestClassifier(random_state=32)
modelo_rf.fit(X_treino_padronizado, y_treino)

previsao_rf = modelo_rf.predict(X_teste_padronizado)
acuracia_rf = accuracy_score(y_teste, previsao_rf)

print(f"Acurácia Random Forest: {acuracia_rf}")